In [1]:
import cv2
import numpy as np
import autopy
import mediapipe as mp
import time
import math


In [ ]:
hand_landmarks = {
    0: "WRIST",
    1: "THUMB_CMC",
    2: "THUMB_MCP",
    3: "THUMB_IP",
    4: "THUMB_TIP",
    5: "INDEX_FINGER_MCP",
    6: "INDEX_FINGER_PIP",
    7: "INDEX_FINGER_DIP",
    8: "INDEX_FINGER_TIP",
    9: "MIDDLE_FINGER_MCP",
    10: "MIDDLE_FINGER_PIP",
    11: "MIDDLE_FINGER_DIP",
    12: "MIDDLE_FINGER_TIP",
    13: "RING_FINGER_MCP",
    14: "RING_FINGER_PIP",
    15: "RING_FINGER_DIP",
    16: "RING_FINGER_TIP",
    17: "PINKY_MCP",
    18: "PINKY_PIP",
    19: "PINKY_DIP",
    20: "PINKY_TIP"
}

In [2]:


def handlandmarks(image):
    landmarks=[]

    image_rgb=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image_rgb.flags.writeable=False
    results=hands.process(image_rgb)
    image_rgb.flags.writeable=True

    if results.multi_hand_landmarks:
        handLms=results.multi_hand_landmarks[0]

        mpDraw.draw_landmarks(
            image,
            handLms,
            mp.solutions.hands.HAND_CONNECTIONS,
            landmark_style,
            connection_style
        )

        h,w,c=image.shape

        for idx,lm in enumerate(handLms.landmark):
            cx,cy=int(lm.x*w),int(lm.y*h)
            landmarks.append([idx,cx,cy])

    return landmarks



In [3]:

def fingers(landmarks):
    tipIds=[4,8,12,16,20]
    fingers=[]

    if landmarks[tipIds[0]][1]>landmarks[tipIds[0]-1][1]:
        fingers.append(1)
    else:
        fingers.append(0)

    for id in range(1,5):
        if landmarks[tipIds[id]][2]<landmarks[tipIds[id]-2][2]:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers



In [28]:

cap=cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,1080)
cap.set(cv2.CAP_PROP_FPS,30)

hands=mp.solutions.hands.Hands(static_image_mode=False,max_num_hands=1,min_detection_confidence=0.5,min_tracking_confidence=0.5)

mpDraw=mp.solutions.drawing_utils
landmark_style=mpDraw.DrawingSpec(color=(255,255,255),thickness=2,circle_radius=4)
connection_style=mpDraw.DrawingSpec(color=(255,180,0),thickness=3,circle_radius=2)

screen_w,screen_h=autopy.screen.size()
screen_w+=100
screen_h+=100
click=False
double_click=False

prev_time=time.time()
fps=0

while True:

    ret,frame=cap.read()

    if not ret:
        break

    frame=cv2.flip(frame,-1)

    landmarks=handlandmarks(frame)

    if len(landmarks)!=0:

        # Index
        x1,y1=landmarks[8][1:]

        # Middle
        x2,y2=landmarks[12][1:]

        # Thumb
        x3,y3=landmarks[4][1:]

        finger=fingers(landmarks)

        # Mouse movement
        if finger[1]==1 and finger[2]==0:

            x4=np.interp(x1,(100,1820),(0,screen_w))
            y4=np.interp(y1,(100,980),(0,screen_h))
            x4=max(0,min(int(x4),screen_w-1))
            y4=max(0,min(int(y4),screen_h-1))

            autopy.mouse.move(x4,y4)

            cv2.putText(frame,"MOVE",(x1-50,y1-30),
                        cv2.FONT_HERSHEY_PLAIN,2,(0,255,0),3)

 
        # Click
        if finger[1]==1 and finger[2]==1:
        
            length=math.hypot(x2-x1,y2-y1)
        
            cv2.line(frame,(x1,y1),(x2,y2),(0,255,0),3)
        
            # Single click
            if length<70 and click==False:
        
                autopy.mouse.click()
        
                click=True
        
                cv2.putText(frame,"CLICK",(x1-50,y1-30),
                    cv2.FONT_HERSHEY_PLAIN,2,(0,255,0),3)
        
            # Release
            if length>100:
        
                click=False
        
        
        # Double click
        if finger[1]==1 and finger[2]==1:
        
            length=math.hypot(x2-x1,y2-y1)
        
            if length<70 and double_click==False:
        
                autopy.mouse.click()
                autopy.mouse.click()
        
                double_click=True
        
                cv2.putText(frame,"DOUBLE CLICK",(x1-100,y1-30),
                            cv2.FONT_HERSHEY_PLAIN,2,(255,0,255),3)
        
            if length>100:
        
                double_click=False
 

    current_time=time.time()
    fps=1/(current_time-prev_time)
    prev_time=current_time

    cv2.putText(frame,f"FPS: {fps:.1f}",(10,50),
                cv2.FONT_HERSHEY_PLAIN,3,(255,180,0),3)

    cv2.imshow("Virtual Mouse",frame)

    if cv2.waitKey(1)==ord("q"):
        break

cap.release()
hands.close()
cv2.destroyAllWindows()

